# 04 — Autoencoder: Colab setup and data loading

This notebook sets up the environment and loads the **already-processed FMA feature vectors** for autoencoder training. It **does not** repeat preprocessing or define/train a model. Run cells in order. The `.npz` files must be committed to the repository at `data/processed/`.

## 1. Repository setup

Set `REPO_URL` to your GitHub repository's HTTPS clone URL. In Colab the repository will be cloned into `/content`. Locally, the notebook searches its parent directories for `data/processed/`. For a private GitHub repository, authenticate in Colab before cloning; do not paste credentials into the notebook.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_NAME = "music-similarity-representation-learning"
REPO_URL = "https://github.com/samiyaalizaidi/music-similarity-representation-learning.git"  # EDIT ME

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    project_root = Path("/content") / REPO_NAME
    if not project_root.exists():
        if "YOUR_USERNAME" in REPO_URL:
            raise ValueError("Replace YOUR_USERNAME in REPO_URL with your GitHub username.")
        subprocess.run(["git", "clone", REPO_URL, str(project_root)], check=True)
else:
    cwd = Path.cwd().resolve()
    project_root = next(
        (p for p in (cwd, *cwd.parents) if (p / "data" / "processed").is_dir()),
        None,
    )
    if project_root is None:
        raise FileNotFoundError("Run this notebook inside your cloned project repository.")

PROJECT_ROOT = project_root.resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
print("Project root:", PROJECT_ROOT)
print("Processed data:", DATA_DIR)
print("Files:", sorted(p.name for p in DATA_DIR.glob("*.npz")))

If your repository was already cloned in this Colab session **before** the `.npz` files were pushed, run the optional cell below to pull updates.

In [ ]:
# OPTIONAL: Uncomment to update an existing clean clone.
# subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)

## 2. Import dependencies and select device

Colab normally includes NumPy and PyTorch. If an import fails, install the missing package in a separate cell.

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

## 3. Load train / validation / test arrays

Each archive is expected to contain `X` (standardized numeric features) and `track_ids`. This cell checks the contract rather than silently guessing alternative keys.

In [ ]:
def load_split(name: str):
    path = DATA_DIR / f"{name}.npz"
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Check your GitHub commit and clone.")
    with np.load(path, allow_pickle=False) as archive:
        required = {"X", "track_ids"}
        missing = required - set(archive.files)
        if missing:
            raise KeyError(f"{path.name} missing keys {sorted(missing)}; found {archive.files}")
        X = np.asarray(archive["X"], dtype=np.float32)
        ids = np.asarray(archive["track_ids"])
    if X.ndim != 2:
        raise ValueError(f"{name}: expected 2D X, got {X.shape}")
    if len(ids) != len(X):
        raise ValueError(f"{name}: track_ids and X have different lengths")
    if not np.isfinite(X).all():
        raise ValueError(f"{name}: X contains NaN or infinite values")
    return X, ids

X_train, train_ids = load_split("training")
X_val, val_ids = load_split("validation")
X_test, test_ids = load_split("test")

INPUT_DIM = X_train.shape[1]
if X_val.shape[1] != INPUT_DIM or X_test.shape[1] != INPUT_DIM:
    raise ValueError("Feature dimensions differ across splits")

for name, X, ids in [("train", X_train, train_ids), ("val", X_val, val_ids), ("test", X_test, test_ids)]:
    print(f"{name:5s} X={X.shape} dtype={X.dtype}, track IDs={len(ids)}")
print("Autoencoder input dimension:", INPUT_DIM)

## 4. Create PyTorch DataLoaders

Autoencoders reconstruct their input, so each dataset yields `(features, features)`. The loaders stay on CPU; training code can move batches to `DEVICE`. Track IDs remain in separate arrays for downstream song retrieval/evaluation.

In [ ]:
def make_loader(X: np.ndarray, *, shuffle: bool) -> DataLoader:
    tensor = torch.from_numpy(X)
    return DataLoader(
        TensorDataset(tensor, tensor),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
    )

train_loader = make_loader(X_train, shuffle=True)
val_loader = make_loader(X_val, shuffle=False)
test_loader = make_loader(X_test, shuffle=False)

inputs, targets = next(iter(train_loader))
assert torch.equal(inputs, targets)
assert inputs.shape[1] == INPUT_DIM
print("First batch:", inputs.shape)
print("Targets equal inputs:", torch.equal(inputs, targets))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

## 5. Ready for model development

Your teammate can now define a model with input size `INPUT_DIM`, train using `train_loader`, validate with `val_loader`, and preserve `train_ids`, `val_ids`, and `test_ids` when exporting embeddings. **Do not fit another scaler or mix test examples into training.**

Example training-loop batch unpacking (not executed here):

```python
for features, targets in train_loader:
    features = features.to(DEVICE)
    targets = targets.to(DEVICE)
    # predictions = model(features)
    # loss = criterion(predictions, targets)
```
